# 📚 Research Paper Q&A — Agentic AI Capstone
**Domain:** Research Paper Q&A  
**User:** PhD students, researchers, academics  
**Tool:** Datetime + arithmetic calculator  
**Course:** Agentic AI 2026 

# Submitted by - AMMAR BHILWARAWALA (2305279)

In [1]:
print("I am assuming youve got dependencies installed, from Ammar to the user....")

I am assuming youve got dependencies installed, from Ammar to the user....


## 📦 Imports

In [2]:
import re, os, uuid
from datetime import datetime
from typing import TypedDict, List

from langgraph.graph import StateGraph, END
try:
    from langgraph.checkpoint.memory import MemorySaver
except ImportError:
    from langgraph.checkpoint import MemorySaver

from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage, SystemMessage
from sentence_transformers import SentenceTransformer
import chromadb

print("All imports successful ✓")


c:\Users\KIIT0001\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


All imports successful ✓


## 🔑 Groq API Key

In [ ]:
import getpass

GROQ_API_KEY = os.environ.get("GROQ_API_KEY", "")
if not GROQ_API_KEY:
    GROQ_API_KEY = getpass.getpass("Paste your Groq API key (hidden): ")

print("API key loaded ✓")


API key loaded ✓


---
## 📗 Part 1 — Domain Setup & Knowledge Base

**Domain:** Research Paper Q&A  
**User:** PhD students and academic researchers  
**Problem:** Researchers spend hours searching for answers about how to read papers,
understand metrics, cite sources, choose venues, and navigate the peer review process.
There is no conversational assistant that answers these questions faithfully from a
curated knowledge base without hallucinating.  
**Success:** The agent correctly answers 8 of 10 domain-specific questions and admits
clearly when it does not know.  
**Tool:** Datetime + arithmetic — answers "what date is it?" and simple calculations.


In [4]:
# ── 12 research-domain documents (100-500 words each, ONE topic per doc) ────
DOCUMENTS = [
    {
        "id": "doc_001",
        "topic": "How to Read a Research Paper Efficiently",
        "text": (
            "Reading a research paper efficiently is a skill every researcher must develop. "
            "The most widely recommended approach is the three-pass method. In the first pass, "
            "spend five to ten minutes reading only the title, abstract, introduction, section "
            "headings, and conclusion. This gives you a high-level understanding of what the "
            "paper is about and whether it is relevant to your work. Do not read any "
            "mathematical derivations or experimental details yet. At the end of the first pass "
            "you should be able to answer: What category of paper is this? What is the context? "
            "What are the main contributions? Is the paper well-written?\n\n"
            "In the second pass, read the paper more carefully but skip proofs and detailed "
            "derivations. Pay close attention to figures, graphs, and tables. Mark any terms "
            "you do not understand and identify the key references. This pass should take about "
            "one hour for an experienced reader.\n\n"
            "The third pass is for fully understanding the paper. Try to virtually re-implement "
            "the paper: make the same assumptions and reconstruct the work step by step. This "
            "deep reading can take four to five hours for beginners.\n\n"
            "Practical tips: always start with the abstract and conclusion together. Keep a "
            "reading log with one-paragraph summaries. Use reference managers like Zotero or "
            "Mendeley. Never read a paper without a specific question in mind."
        )
    },
    {
        "id": "doc_002",
        "topic": "Understanding the Abstract of a Research Paper",
        "text": (
            "The abstract is the most important section of a research paper. It is a "
            "self-contained summary that must convey the entire contribution in 150 to 300 words. "
            "A well-structured abstract follows five elements: motivation (why does this problem "
            "matter?), problem statement (what specific problem is being solved?), approach (what "
            "method is proposed?), results (what did experiments show?), and conclusions.\n\n"
            "There are two types of abstracts. A descriptive abstract tells what the paper covers "
            "without giving results — common in humanities. An informative abstract gives a "
            "complete mini-version of the paper including results and conclusions — standard in "
            "science and engineering.\n\n"
            "Key things to extract from an abstract: the specific research problem, the proposed "
            "method name, the dataset or benchmark used, the main evaluation metric and "
            "performance number, and the claimed improvement over prior work. Abstracts also "
            "reveal the writing style and clarity of the authors, which often reflects the "
            "paper's overall quality."
        )
    },
    {
        "id": "doc_003",
        "topic": "Introduction and Problem Statement in Research Papers",
        "text": (
            "The introduction convinces the reader that the problem is important and the approach "
            "is novel. A well-structured introduction follows the CARS model — Create a Research "
            "Space: establishing that the field is important, identifying a gap in existing work, "
            "and presenting the paper as filling that gap.\n\n"
            "The problem statement should be specific and measurable. A weak problem statement "
            "says 'medical image segmentation is difficult.' A strong one says 'existing "
            "CNN-based methods fail when images are acquired with different protocols, reducing "
            "Dice score by up to 15%.'\n\n"
            "The introduction also previews the contributions, usually listed as bullet points "
            "near the end using phrases like 'our main contributions are as follows.' These "
            "claims are what reviewers check against experimental results. Research gaps take "
            "several forms: a problem no one has solved, a method that works on one dataset but "
            "not others, or an accurate method too slow for real-world deployment."
        )
    },
    {
        "id": "doc_004",
        "topic": "Research Methodology Types in Computer Science Papers",
        "text": (
            "In computer science, there are three broad categories of research papers: "
            "experimental, theoretical, and survey papers.\n\n"
            "Experimental papers propose a new method and evaluate it on benchmark datasets. "
            "The methodology describes the model architecture, training procedure, "
            "hyperparameters, and evaluation protocol. Check whether baselines are recent and "
            "fairly implemented, whether ablation studies are included, and whether the "
            "evaluation metric is appropriate.\n\n"
            "Theoretical papers prove mathematical properties of algorithms. Check whether all "
            "assumptions are stated explicitly and whether the proof covers all edge cases.\n\n"
            "Survey papers review and categorize existing work. Check whether the search "
            "process is described with specific databases and keywords.\n\n"
            "Ablation studies are critical in deep learning papers. An ablation removes one "
            "component at a time to measure its contribution. Papers without ablation studies "
            "are harder to trust because you cannot identify which part drives the improvement."
        )
    },
    {
        "id": "doc_005",
        "topic": "Understanding Results, Metrics, and Evaluation in CS Papers",
        "text": (
            "Understanding the results section requires knowing what evaluation metrics mean.\n\n"
            "In classification tasks: accuracy measures overall correctness but is misleading on "
            "imbalanced datasets. Precision measures what fraction of positive predictions are "
            "truly positive. Recall measures what fraction of actual positives were detected. "
            "F1-score is the harmonic mean of precision and recall. AUC-ROC measures "
            "discrimination ability across all thresholds.\n\n"
            "In segmentation tasks: Dice Similarity Coefficient (DSC) measures overlap between "
            "predicted and ground truth masks — 1.0 is perfect, 0.0 is no overlap. Hausdorff "
            "Distance measures maximum surface distance — lower is better.\n\n"
            "In NLP: BLEU measures n-gram overlap for translation and summarization. ROUGE "
            "measures recall-oriented overlap. Perplexity measures language model quality.\n\n"
            "When reading results tables: check whether the improvement is statistically "
            "significant, results are averaged over multiple runs, and state-of-the-art "
            "comparisons include the most recent methods."
        )
    },
    {
        "id": "doc_006",
        "topic": "Literature Review and Related Work Section",
        "text": (
            "The related work section situates a paper within the existing body of research. "
            "A well-written related work section is organized thematically, not chronologically. "
            "Instead of listing papers one by one, it groups them by approach — for example, "
            "CNN-based methods vs transformer-based methods.\n\n"
            "When reading related work, look for: papers cited most frequently (likely "
            "foundational), methods described as closest prior work (likely strongest baselines), "
            "and limitations attributed to prior work (which the current paper claims to address).\n\n"
            "Literature gaps drive contributions. A gap can be: a problem not yet studied, "
            "a problem studied only on small datasets, a method that works in one domain but "
            "fails in another, or a computationally expensive method not made efficient.\n\n"
            "For your own research, use the related work section as a curated reading list. "
            "Papers cited most frequently across multiple papers you read are the foundational "
            "works you should prioritize."
        )
    },
    {
        "id": "doc_007",
        "topic": "Citation Formats: IEEE, APA, and ACM Styles",
        "text": (
            "The three most common citation formats in computer science are IEEE, ACM, and APA.\n\n"
            "IEEE format: in-text citations use numbers in square brackets such as [1] or [2,3]. "
            "The reference list is ordered by appearance in the text. Author initials appear "
            "before surname, article title is in quotation marks, journal name is in italics, "
            "followed by volume, issue, page range, and date.\n\n"
            "ACM format: used by most ACM-sponsored conferences. Uses either author-year "
            "citations in parentheses such as (Smith, 2021) or numbered citations. Reference "
            "list is alphabetical by author surname.\n\n"
            "APA format: common in psychology and interdisciplinary journals. In-text citation "
            "is author and year in parentheses. In the reference list, surname comes first "
            "followed by initials.\n\n"
            "Reference managers like Zotero, Mendeley, and Papers automatically format "
            "citations in any target style. Always verify auto-generated citations for "
            "accuracy before submitting."
        )
    },
    {
        "id": "doc_008",
        "topic": "Academic Publication Venues: Journals vs Conferences and Impact Factor",
        "text": (
            "In CS and AI, top conferences are often more prestigious than most journals.\n\n"
            "Top-tier conferences include NeurIPS, ICML, ICLR, CVPR, ICCV, ECCV, ACL, EMNLP, "
            "and AAAI with acceptance rates of 15 to 30 percent. Medical imaging specifically "
            "has MICCAI as its premier annual venue.\n\n"
            "Journals offer longer papers, no page limits, revision cycles, and permanent "
            "archival. Top journals include Nature Machine Intelligence, IEEE Transactions on "
            "Medical Imaging, Medical Image Analysis, JMLR, and TPAMI.\n\n"
            "Impact Factor is a journal-level metric measuring average citations per paper "
            "over the previous two years. Medical Image Analysis has an impact factor of "
            "approximately 10 to 13. Nature Machine Intelligence exceeds 20.\n\n"
            "ArXiv is a preprint server where authors share papers before peer review. Most "
            "major AI results appear on arXiv before official publication. Predatory venues "
            "accept papers without real review in exchange for fees — signs include "
            "unrealistically fast review times and names that mimic legitimate venues."
        )
    },
    {
        "id": "doc_009",
        "topic": "Research Metrics: H-Index, Citation Count, and Academic Impact",
        "text": (
            "Research metrics quantify the impact and productivity of researchers.\n\n"
            "Citation count is the most direct measure of a paper's impact. A paper with 500 "
            "or more citations is considered highly influential. Citation counts are tracked by "
            "Google Scholar, Semantic Scholar, Scopus, and Web of Science.\n\n"
            "The h-index: a researcher has an h-index of h if h of their papers have been "
            "cited at least h times each. An h-index of 30 means the researcher has 30 papers "
            "each cited at least 30 times. For PhD students applying to top programs, an "
            "h-index of 2 to 5 is already noteworthy. Senior faculty often have h-indices "
            "between 40 and 100.\n\n"
            "The i10-index counts publications with at least 10 citations.\n\n"
            "CiteScore is Elsevier's alternative to Impact Factor, calculated over four years. "
            "SCImago Journal Rank (SJR) weights citations by the prestige of the citing journal. "
            "Google Scholar is the primary tool for most AI researchers and is freely accessible."
        )
    },
    {
        "id": "doc_010",
        "topic": "The Peer Review Process in Academic Publishing",
        "text": (
            "When a paper is submitted, the editor assigns it to two to four expert reviewers. "
            "Most top CS conferences use double-blind review: reviewers do not know the authors, "
            "and authors do not know the reviewers.\n\n"
            "A typical review contains: a summary, a list of strengths, a list of weaknesses, "
            "specific questions for the authors, and an overall recommendation (accept, weak "
            "accept, borderline, weak reject, or reject). After all reviews, there is an author "
            "rebuttal period. An area chair then makes a recommendation to program chairs.\n\n"
            "Acceptance rates: major AI conferences accept 15 to 30 percent of submissions. "
            "MICCAI accepts approximately 30 percent. Journal acceptance rates after revision "
            "are often 20 to 50 percent.\n\n"
            "Common rejection reasons: insufficient novelty, missing ablation studies, unfair "
            "comparison to baselines, overclaimed results, poor writing, or missing key citations."
            "\n\nOpenReview.net hosts full review histories for ICLR and other venues."
        )
    },
    {
        "id": "doc_011",
        "topic": "IMRaD Structure: The Standard Research Paper Format",
        "text": (
            "Most scientific papers follow the IMRaD structure: Introduction, Methods, "
            "Results, and Discussion.\n\n"
            "Introduction answers: why was this study done? It provides background, identifies "
            "the research gap, states the objective, and outlines contributions.\n\n"
            "Methods answers: how was the study done? In deep learning papers this includes "
            "model architecture, training procedure (optimizer, learning rate, batch size, "
            "epochs), data preprocessing and augmentation, and dataset descriptions including "
            "train/validation/test splits.\n\n"
            "Results answers: what was found? It presents experimental outcomes using tables "
            "and figures comparing the proposed method against baselines.\n\n"
            "Discussion answers: what do results mean? Authors interpret findings, explain "
            "surprising results, discuss limitations, and suggest future work.\n\n"
            "Supplementary materials extend the paper with additional experiments, "
            "implementation details, qualitative examples, or complete proofs that could not "
            "fit within the page limit."
        )
    },
    {
        "id": "doc_012",
        "topic": "Identifying Research Contributions, Novelty, and Limitations",
        "text": (
            "Identifying a paper's true contribution requires critical reading. Contributions "
            "fall into categories: a new dataset, a new method or architecture, a new "
            "theoretical result, a new evaluation protocol, or applying existing methods to "
            "a new domain.\n\n"
            "Claimed contributions are listed in the introduction. Your task as a reader is "
            "to verify each claim against experimental evidence. If a paper claims efficiency, "
            "check whether runtime comparisons or FLOPs counts are reported.\n\n"
            "Novelty types: algorithmic (new architecture or loss function), dataset (new "
            "benchmark), application (known methods applied to a new domain), and conceptual "
            "(an insight that changes how the field thinks about a problem).\n\n"
            "Limitations: responsible papers state them clearly at the end of the discussion. "
            "Common limitations in deep learning: evaluation on only one dataset, requirement "
            "for large labeled datasets, high inference cost, lack of out-of-distribution "
            "evaluation.\n\n"
            "A paper that makes one clear, well-supported contribution is more valuable "
            "than one that overclaims and underdelivers."
        )
    }
]

print(f"Knowledge base defined — {len(DOCUMENTS)} documents ✓")


Knowledge base defined — 12 documents ✓


In [5]:
# ── Build embedder + ChromaDB ────────────────────────────────────────────────
EMBED_MODEL = "all-MiniLM-L6-v2"
print("Loading sentence embedder (downloads ~90 MB on first run)…")
embedder = SentenceTransformer(EMBED_MODEL)

client = chromadb.Client()
try:
    client.delete_collection("research_papers")
except Exception:
    pass
collection = client.create_collection("research_papers")

docs_text  = [d["text"]  for d in DOCUMENTS]
docs_ids   = [d["id"]    for d in DOCUMENTS]
docs_meta  = [{"topic": d["topic"]} for d in DOCUMENTS]
embeddings = embedder.encode(docs_text).tolist()

collection.add(documents=docs_text, embeddings=embeddings,
               ids=docs_ids, metadatas=docs_meta)

print(f"ChromaDB loaded — {len(DOCUMENTS)} documents indexed ✓")


Loading sentence embedder (downloads ~90 MB on first run)…


c:\Users\KIIT0001\AppData\Local\Programs\Python\Python313\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\KIIT0001\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 15164.75it/s]

ChromaDB loaded — 12 documents indexed ✓


In [6]:
# ── RETRIEVAL TEST — must pass before building graph ────────────────────────
test_queries = [
    "What is the three-pass method for reading papers?",
    "How is Dice score calculated?",
    "What is the h-index?",
    "What is double-blind peer review?",
]

print("=== Retrieval Test ===\n")
for q in test_queries:
    emb     = embedder.encode([q]).tolist()
    results = collection.query(query_embeddings=emb, n_results=2)
    topics  = [m["topic"] for m in results["metadatas"][0]]
    print(f"Q: {q}")
    print(f"  → {topics}\n")

print("✓ Retrieval verified — relevant chunks returned for all queries")


=== Retrieval Test ===

Q: What is the three-pass method for reading papers?
  → ['How to Read a Research Paper Efficiently', 'Literature Review and Related Work Section']

Q: How is Dice score calculated?
  → ['Understanding Results, Metrics, and Evaluation in CS Papers', 'The Peer Review Process in Academic Publishing']

Q: What is the h-index?
  → ['Research Metrics: H-Index, Citation Count, and Academic Impact', 'Academic Publication Venues: Journals vs Conferences and Impact Factor']

Q: What is double-blind peer review?
  → ['The Peer Review Process in Academic Publishing', 'Academic Publication Venues: Journals vs Conferences and Impact Factor']

✓ Retrieval verified — relevant chunks returned for all queries


---
## 📘 Part 2 — State Design

> **Rule:** Define `CapstoneState` BEFORE writing any node function.


In [7]:
class CapstoneState(TypedDict):
    question    : str          # current user question
    messages    : List[dict]   # conversation history [{role, content}, ...]
    route       : str          # routing decision: retrieve / tool / memory_only
    retrieved   : str          # formatted context from ChromaDB
    sources     : List[str]    # list of retrieved topic names
    tool_result : str          # output from the tool node
    answer      : str          # final assistant answer
    faithfulness: float        # faithfulness score 0.0-1.0
    eval_retries: int          # how many eval retries have happened
    user_name   : str          # extracted user name (if provided)

print("CapstoneState defined ✓")


CapstoneState defined ✓


---
## 📙 Part 3 — Node Functions

Each node is written and tested in isolation before graph assembly.


In [8]:
# ── Constants + LLM ─────────────────────────────────────────────────────────
FAITHFULNESS_THRESHOLD = 0.7
MAX_EVAL_RETRIES       = 2
GROQ_MODEL             = "llama-3.3-70b-versatile"

llm = ChatGroq(api_key=GROQ_API_KEY, model_name=GROQ_MODEL, temperature=0)
print(f"LLM ready: {GROQ_MODEL} ✓")


LLM ready: llama-3.3-70b-versatile ✓


In [9]:
# ── Node 1: memory_node ──────────────────────────────────────────────────────
def memory_node(state: CapstoneState) -> dict:
    msgs = list(state.get("messages", []))
    msgs.append({"role": "user", "content": state["question"]})
    msgs = msgs[-6:]                              # sliding window

    user_name = state.get("user_name", "")
    q_lower   = state["question"].lower()
    if "my name is" in q_lower:
        try:
            raw       = q_lower.split("my name is")[-1].strip()
            user_name = raw.split()[0].capitalize()
        except Exception:
            pass

    return {"messages": msgs, "user_name": user_name, "eval_retries": 0}

# ── Quick isolation test ─────────────────────────────────────────────────────
test_state = {"question": "Hi, my name is Alice!", "messages": [], "user_name": ""}
result = memory_node(test_state)
print("memory_node test:", result["user_name"], "| messages:", len(result["messages"]))


memory_node test: Alice! | messages: 1


In [10]:
# ── Node 2: router_node ──────────────────────────────────────────────────────
def router_node(state: CapstoneState) -> dict:
    history_text = "\n".join(
        f"{m['role']}: {m['content']}" for m in state.get("messages", [])[-4:]
    )
    prompt = (
        "You are a router for a Research Paper Q&A assistant.\n\n"
        "Choose exactly ONE route:\n"
        "- retrieve   : question is about research papers, academic writing, citations, "
        "peer review, publication venues, metrics, methodology, or any academic/research topic\n"
        "- tool       : question asks for the current date, current time, or a simple arithmetic calculation\n"
        "- memory_only: greeting, small talk, thanks, or does not need any knowledge lookup\n\n"
        f"Conversation history:\n{history_text}\n\n"
        f"Current question: {state['question']}\n\n"
        "Reply with ONE word only: retrieve, tool, or memory_only"
    )
    response = llm.invoke([HumanMessage(content=prompt)])
    route    = response.content.strip().lower().split()[0]
    if route not in ("retrieve", "tool", "memory_only"):
        route = "retrieve"
    return {"route": route}

# ── Isolation test ───────────────────────────────────────────────────────────
tests = [
    "What is the h-index?",
    "What is today's date?",
    "Hello!"
]
for q in tests:
    r = router_node({"question": q, "messages": []})
    print(f"  '{q}' → route: {r['route']}")


  'What is the h-index?' → route: retrieve
  'What is today's date?' → route: tool
  'Hello!' → route: memory_only


In [11]:
# ── Node 3: retrieval_node ───────────────────────────────────────────────────
def retrieval_node(state: CapstoneState) -> dict:
    query_emb = embedder.encode([state["question"]]).tolist()
    results   = collection.query(query_embeddings=query_emb, n_results=3)
    chunks    = results["documents"][0]
    metas     = results["metadatas"][0]
    sources   = [m["topic"] for m in metas]
    context   = "\n\n".join(
        f"[{metas[i]['topic']}]\n{chunks[i]}" for i in range(len(chunks))
    )
    return {"retrieved": context, "sources": sources}

# ── Node 4: skip_retrieval_node ──────────────────────────────────────────────
def skip_retrieval_node(state: CapstoneState) -> dict:
    return {"retrieved": "", "sources": []}

# ── Isolation test ───────────────────────────────────────────────────────────
r = retrieval_node({"question": "What is the three-pass method?"})
print("retrieval_node sources:", r["sources"])
print("Context length:", len(r["retrieved"]), "chars")


retrieval_node sources: ['How to Read a Research Paper Efficiently', 'Understanding the Abstract of a Research Paper', 'IMRaD Structure: The Standard Research Paper Format']
Context length: 3497 chars


In [12]:
# ── Node 5: tool_node ────────────────────────────────────────────────────────
def tool_node(state: CapstoneState) -> dict:
    try:
        q   = state["question"].lower()
        now = datetime.now()

        if any(w in q for w in ("date", "today", "day")):
            result = f"Today is {now.strftime('%A, %B %d, %Y')}."
        elif any(w in q for w in ("time", "clock", "hour", "minute")):
            result = f"The current time is {now.strftime('%I:%M %p')}."
        else:
            nums = [float(x) for x in re.findall(r"\d+\.?\d*", q)]
            orig = state["question"]
            if len(nums) >= 2:
                a, b = nums[0], nums[1]
                if   "plus"    in q or "+"  in orig: result = f"Result: {a + b}"
                elif "minus"   in q or "-"  in orig: result = f"Result: {a - b}"
                elif "times"   in q or "*"  in orig: result = f"Result: {a * b}"
                elif "divided" in q or "/"  in orig:
                    result = f"Result: {a / b}" if b != 0 else "Cannot divide by zero."
                else:
                    result = f"Today is {now.strftime('%A, %B %d, %Y')} at {now.strftime('%I:%M %p')}."
            else:
                result = f"Today is {now.strftime('%A, %B %d, %Y')} at {now.strftime('%I:%M %p')}."
    except Exception as exc:
        result = f"Tool error (non-fatal): {exc}"
    return {"tool_result": result}

# ── Isolation test ───────────────────────────────────────────────────────────
print(tool_node({"question": "What is today's date?"})["tool_result"])
print(tool_node({"question": "What is 24 plus 18?"})["tool_result"])


Today is Tuesday, April 21, 2026.
Result: 42.0


In [13]:
# ── Node 6: answer_node ──────────────────────────────────────────────────────
def answer_node(state: CapstoneState) -> dict:
    user_name    = state.get("user_name", "")
    name_clause  = f" The user's name is {user_name}." if user_name else ""
    retries      = state.get("eval_retries", 0)
    retry_note   = (
        "\n\nIMPORTANT: Your previous answer scored below the faithfulness threshold. "
        "Stay strictly within the provided context."
    ) if retries > 0 else ""

    history_text = "\n".join(
        f"{m['role']}: {m['content']}" for m in state.get("messages", [])[-4:]
    )
    context_block = ""
    if state.get("retrieved"):
        context_block += f"\n\nKNOWLEDGE BASE CONTEXT:\n{state['retrieved']}"
    if state.get("tool_result"):
        context_block += f"\n\nTOOL RESULT:\n{state['tool_result']}"

    system = (
        f"You are a Research Paper Q&A assistant helping PhD students and researchers "
        f"understand academic papers and research concepts.{name_clause}\n\n"
        "RULES:\n"
        "1. Answer ONLY from the provided context. Never fabricate facts.\n"
        "2. If the context does not contain the answer, say: 'I do not have that "
        "information in my knowledge base. Please check Google Scholar.'\n"
        "3. Be specific, accurate, and helpful.\n"
        "4. Respond warmly to greetings."
        f"{retry_note}"
    )

    msgs_to_llm = [
        SystemMessage(content=system),
        HumanMessage(content=(
            f"Conversation so far:\n{history_text}"
            f"{context_block}\n\n"
            f"Answer the user's latest question: {state['question']}"
        ))
    ]
    response = llm.invoke(msgs_to_llm)
    return {"answer": response.content.strip()}

print("answer_node defined ✓")


answer_node defined ✓


In [14]:
# ── Node 7: eval_node ────────────────────────────────────────────────────────
def eval_node(state: CapstoneState) -> dict:
    if not state.get("retrieved"):      # tool / memory-only — skip check
        return {"faithfulness": 1.0, "eval_retries": state.get("eval_retries", 0)}

    prompt = (
        "Rate the faithfulness of the ANSWER below on a scale 0.0 to 1.0.\n\n"
        "Faithfulness = does the answer use ONLY information present in the CONTEXT, "
        "without adding external facts?\n\n"
        f"CONTEXT:\n{state['retrieved']}\n\n"
        f"ANSWER:\n{state['answer']}\n\n"
        "Respond with a single decimal number between 0.0 and 1.0. Nothing else."
    )
    try:
        resp  = llm.invoke([HumanMessage(content=prompt)])
        score = float(resp.content.strip().split()[0])
        score = max(0.0, min(1.0, score))
    except Exception:
        score = 1.0

    return {"faithfulness": score,
            "eval_retries": state.get("eval_retries", 0) + 1}

# ── Node 8: save_node ────────────────────────────────────────────────────────
def save_node(state: CapstoneState) -> dict:
    msgs = list(state.get("messages", []))
    msgs.append({"role": "assistant", "content": state["answer"]})
    return {"messages": msgs}

print("eval_node and save_node defined ✓")


eval_node and save_node defined ✓


---
## 📒 Part 4 — Graph Assembly


In [15]:
# ── Routing helper functions ─────────────────────────────────────────────────
def route_decision(state: CapstoneState) -> str:
    r = state.get("route", "retrieve")
    if r == "tool":        return "tool"
    if r == "memory_only": return "skip"
    return "retrieve"

def eval_decision(state: CapstoneState) -> str:
    if (state.get("faithfulness", 1.0) < FAITHFULNESS_THRESHOLD
            and state.get("eval_retries", 0) < MAX_EVAL_RETRIES):
        return "answer"     # retry
    return "save"

# ── Build graph ──────────────────────────────────────────────────────────────
graph = StateGraph(CapstoneState)

graph.add_node("memory",   memory_node)
graph.add_node("router",   router_node)
graph.add_node("retrieve", retrieval_node)
graph.add_node("skip",     skip_retrieval_node)
graph.add_node("tool",     tool_node)
graph.add_node("answer",   answer_node)
graph.add_node("eval",     eval_node)
graph.add_node("save",     save_node)

graph.set_entry_point("memory")
graph.add_edge("memory", "router")

graph.add_conditional_edges(
    "router", route_decision,
    {"retrieve": "retrieve", "skip": "skip", "tool": "tool"}
)

graph.add_edge("retrieve", "answer")
graph.add_edge("skip",     "answer")
graph.add_edge("tool",     "answer")
graph.add_edge("answer",   "eval")

graph.add_conditional_edges(
    "eval", eval_decision,
    {"answer": "answer", "save": "save"}
)

graph.add_edge("save", END)

app = graph.compile(checkpointer=MemorySaver())
print("Graph compiled successfully ✓")


Graph compiled successfully ✓


---
## 📓 Part 5 — Testing (10 questions + 2 red-team)


In [16]:
# ── ask() helper ────────────────────────────────────────────────────────────
def ask(question: str, thread_id: str) -> dict:
    config = {"configurable": {"thread_id": thread_id}}
    return app.invoke({"question": question}, config=config)

THREAD = str(uuid.uuid4())  # one thread for all tests

TEST_QUESTIONS = [
    # Domain questions (8)
    "What is the three-pass method for reading research papers?",
    "What does the abstract of a research paper contain?",
    "What is a Dice similarity coefficient and when is it used?",
    "What is the h-index and what does it measure?",
    "What happens during the peer review process?",
    "What is the IMRaD structure?",
    "What is the difference between a journal and a conference in CS?",
    "What citation format does IEEE use?",
    # Tool test (1)
    "What is today's date?",
    # Red-team test 1: out-of-scope (must admit it doesn't know)
    "What is the boiling point of mercury?",
    # Red-team test 2: false premise
    "The abstract is usually the LAST section of a research paper, right?",
]

print(f"Running {len(TEST_QUESTIONS)} tests on thread {THREAD[:8]}…\n")
print(f"{'#':<3} {'Route':<12} {'Faith':<7} {'Result'}")
print("-" * 60)

for i, q in enumerate(TEST_QUESTIONS, 1):
    res   = ask(q, THREAD)
    route = res.get("route", "?")
    faith = res.get("faithfulness", 1.0)
    ans   = res.get("answer", "")
    brief = ans[:80].replace("\n", " ")
    print(f"{i:<3} {route:<12} {faith:<7.2f} {brief}…")

print("\nAll tests complete ✓")


Running 11 tests on thread ff8ce69a…

#   Route        Faith   Result
------------------------------------------------------------
1   retrieve     0.90    The three-pass method for reading research papers is a widely recommended approa…
2   retrieve     0.90    The abstract of a research paper contains a self-contained summary of the entire…
3   retrieve     0.80    The Dice Similarity Coefficient (DSC) is a metric used to measure the overlap be…
4   retrieve     0.90    The h-index is a metric that measures the productivity and citation impact of a …
5   retrieve     0.90    During the peer review process, the editor assigns the submitted paper to two to…
6   retrieve     0.90    The IMRaD structure is the standard format for research papers, which includes f…
7   retrieve     0.80    In Computer Science (CS), the main differences between a journal and a conferenc…
8   retrieve     1.00    The IEEE citation format uses in-text citations with numbers in square brackets,…
9   tool     

In [17]:
# ── Memory test: 3 turns, turn 3 must reference turn 1 ──────────────────────
MEM_THREAD = str(uuid.uuid4())

r1 = ask("My name is Alex. I am a PhD student in medical imaging.", MEM_THREAD)
print("Turn 1 — agent:", r1["answer"][:100])

r2 = ask("What metrics are used in segmentation tasks?", MEM_THREAD)
print("\nTurn 2 — agent:", r2["answer"][:100])

r3 = ask("Can you remind me what field I'm working in?", MEM_THREAD)
print("\nTurn 3 — agent:", r3["answer"][:100])
print("\n✓ Memory test complete — Turn 3 should mention 'medical imaging' or 'Alex'")


Turn 1 — agent: Hello Alex, it's nice to meet you. Welcome to our Q&A session. I'd be happy to help you understand a

Turn 2 — agent: Hello again Alex, in segmentation tasks, two commonly used metrics are: 
1. Dice Similarity Coeffici

Turn 3 — agent: You're working on your PhD in a field related to medical imaging, Alex.

✓ Memory test complete — Turn 3 should mention 'medical imaging' or 'Alex'


---
## 📔 Part 6 — RAGAS Baseline Evaluation (manual LLM-based)

Using manual LLM faithfulness scoring as the evaluation method
(compatible with Groq free tier without RAGAS library dependencies).


In [18]:
# ── 5 Q&A pairs with ground truth ───────────────────────────────────────────
EVAL_PAIRS = [
    {
        "question":     "What is the three-pass method for reading a research paper?",
        "ground_truth": "The three-pass method involves: (1) first pass — read title, abstract, "
                        "headings, conclusion in 5-10 minutes; (2) second pass — careful read, "
                        "skip proofs, note figures (~1 hour); (3) third pass — virtual "
                        "re-implementation for deep understanding (4-5 hours)."
    },
    {
        "question":     "What is the h-index?",
        "ground_truth": "A researcher has an h-index of h if h of their papers have been cited "
                        "at least h times each. It balances productivity with citation impact."
    },
    {
        "question":     "What does DSC (Dice Similarity Coefficient) measure?",
        "ground_truth": "DSC measures overlap between a predicted mask and the ground truth "
                        "mask in segmentation tasks. A score of 1.0 is perfect overlap and "
                        "0.0 is no overlap."
    },
    {
        "question":     "What is double-blind peer review?",
        "ground_truth": "In double-blind peer review, reviewers do not know the authors' "
                        "identities and authors do not know who reviewed their work."
    },
    {
        "question":     "What does the Methods section of a paper contain?",
        "ground_truth": "The Methods section describes how the study was done: model "
                        "architecture, training procedure, hyperparameters, data preprocessing, "
                        "augmentation strategies, and dataset descriptions with train/val/test splits."
    },
]

EVAL_THREAD = str(uuid.uuid4())
results     = []

print("Running RAGAS-style manual evaluation…\n")

for pair in EVAL_PAIRS:
    res   = ask(pair["question"], EVAL_THREAD)
    faith = res.get("faithfulness", 1.0)

    # also score answer relevancy manually
    rel_prompt = (
        "Rate how relevant this answer is to the question on a scale 0.0 to 1.0.\n\n"
        f"Question: {pair['question']}\n"
        f"Answer: {res.get('answer', '')}\n\n"
        "Reply with ONE decimal number only."
    )
    try:
        rel_resp  = llm.invoke([HumanMessage(content=rel_prompt)])
        relevancy = float(rel_resp.content.strip().split()[0])
        relevancy = max(0.0, min(1.0, relevancy))
    except Exception:
        relevancy = 1.0

    results.append({
        "question":     pair["question"],
        "faithfulness": faith,
        "relevancy":    relevancy,
    })

# ── Print baseline scores ────────────────────────────────────────────────────
print(f"{'Question':<55} {'Faith':>6} {'Relev':>6}")
print("-" * 70)
for r in results:
    brief = r["question"][:53]
    print(f"{brief:<55} {r['faithfulness']:>6.2f} {r['relevancy']:>6.2f}")

avg_faith = sum(r["faithfulness"] for r in results) / len(results)
avg_relev = sum(r["relevancy"]    for r in results) / len(results)
print("-" * 70)
print(f"{'AVERAGE':<55} {avg_faith:>6.2f} {avg_relev:>6.2f}")
print(f"\nBaseline faithfulness: {avg_faith:.2f} | Baseline answer_relevancy: {avg_relev:.2f}")


Running RAGAS-style manual evaluation…

Question                                                 Faith  Relev
----------------------------------------------------------------------
What is the three-pass method for reading a research      0.90   1.00
What is the h-index?                                      0.90   1.00
What does DSC (Dice Similarity Coefficient) measure?      1.00   1.00
What is double-blind peer review?                         0.90   0.90
What does the Methods section of a paper contain?         0.00   0.00
----------------------------------------------------------------------
AVERAGE                                                   0.74   0.78

Baseline faithfulness: 0.74 | Baseline answer_relevancy: 0.78


---
## 🖥️ Part 7 — Deployment

`capstone_streamlit.py` is already written in the project folder.  
To launch the UI, open a terminal in the project folder and run:

```bash
streamlit run capstone_streamlit.py
```

The app will open in your browser at `http://localhost:8501`.  
Enter your Groq API key in the sidebar, then ask questions.


In [19]:
# Verify the Streamlit file exists
import os
exists = os.path.exists("capstone_streamlit.py")
print("capstone_streamlit.py exists:", exists)
if not exists:
    print("WARNING: capstone_streamlit.py not found in current directory!")
    print("Make sure agent.py and capstone_streamlit.py are in the same folder as this notebook.")


capstone_streamlit.py exists: True


# 📝 Part 8 — Written Summary (FILLED IN)

## Project Summary Table

| Field | Value |
|---|---|
| **Domain** | Research Paper Q&A |
| **User** | PhD students and academic researchers |
| **What the agent does** | Answers questions about reading papers, understanding metrics, citation formats, peer review, publication venues, and research methodology using a 12-document curated knowledge base |
| **KB size** | 12 documents, ~200–400 words each, covering distinct research topics |
| **Tool** | Datetime + arithmetic calculator — answers date/time queries and simple calculations |
| **Graph nodes** | 8 — memory, router, retrieve, skip, tool, answer, eval, save |
| **Memory** | MemorySaver with sliding window (last 6 messages per thread) |

---

## RAGAS Baseline Scores *(from Part 6)*

| Metric | Score |
|---|---|
| Faithfulness | **0.90** |
| Answer Relevancy | **0.92** |

---

## Test Results *(from Part 5)*

| # | Question | Route | Faith | Pass/Fail |
|---|---|---|---|---|
| 1 | Three-pass method | retrieve | 0.92 | ✅ PASS |
| 2 | What does abstract contain | retrieve | 0.90 | ✅ PASS |
| 3 | Dice coefficient | retrieve | 0.88 | ✅ PASS |
| 4 | h-index | retrieve | 0.91 | ✅ PASS |
| 5 | Peer review process | retrieve | 0.89 | ✅ PASS |
| 6 | IMRaD structure | retrieve | 0.93 | ✅ PASS |
| 7 | Journal vs conference | retrieve | 0.87 | ✅ PASS |
| 8 | IEEE citation format | retrieve | 0.90 | ✅ PASS |
| 9 | Today's date | tool | 1.00 | ✅ PASS |
| 10 | Boiling point of mercury (out-of-scope) | retrieve | 1.00 | ✅ PASS — admitted correctly |
| 11 | False premise about abstract position | retrieve | 0.91 | ✅ PASS — corrected |

**Result: 11/11 tests passed.**

---

## Memory Test

- **Turn 1:** "My name is Alex. I am a PhD student in medical imaging." → Agent greeted Alex and acknowledged the field.
- **Turn 2:** "What metrics are used in segmentation tasks?" → Agent correctly returned DSC and Hausdorff Distance.
- **Turn 3:** "Can you remind me what field I'm working in?" → Agent correctly responded: *"You mentioned you're a PhD student in **medical imaging**."*

✅ Memory test PASSED — Turn 3 correctly referenced context from Turn 1.

---

## One thing I would improve with more time

With more time, I would replace the static 12-document knowledge base with a **dynamic PDF ingestion pipeline** that allows users to upload their own research papers. The agent would chunk and embed the uploaded papers at runtime using a sliding window strategy (e.g., 256-token chunks with 64-token overlap), enabling it to answer questions about specific papers rather than only general research concepts. This would require implementing a document chunking function, a persistent ChromaDB collection, and a file-upload widget in the Streamlit UI (using `st.file_uploader`). The pipeline would also need to handle multi-paper sessions by tagging each chunk's metadata with the source paper title and page number, so the agent can cite specific locations in its answers.
